# 3.0 Inteligencia Artificial Aplicada al Riesgo: Predicción de Transiciones de Fase con XGBoost"

El Notebook 2 culminó con la calibración exitosa de un motor GARCH(1,1) bajo distribución t-Student, certificado en "Zona Verde" por las especificaciones del Comité de Basilea. Sin embargo, los modelos econométricos tradicionales poseen una limitación física crítica: asumen que la volatilidad evoluciona de forma continua y suave en el tiempo, lo que los hace severamente miopes ante Transiciones de Fase repentinas.

Es por ello que se ocupará el GARCH ya que resulta el más indicado para modelar crisis progresivas, pero es incapaz de anticipar el momento exacto en que el mercado va a saltar instantáneamente de un estado de calma a un régimen de pánico debido a choques exógenos cruzados. Para resolver este punto ciego y dotar a la empresa de un sistema de cobertura proactivo, el objetivo de este Notebook 3 es abandonar el modelado puramente paramétrico e implementar un algoritmo de Machine Learning de ensamble no lineal como XGBoost.

El proyecto no va a intentar "adivinar" el precio de mañana usando Machine Learning, lo cual violaría la Hipótesis de Mercados Eficientes. Contraste a ello, el XGBoost se entrenará para resolver un problema de Clasificación Binaria enfocado en predecir regímenes de riesgo extremo:

* La Variable Objetivo (Target / Y): Utilizando la base de datos certificada del Notebook 2, identificaremos los días donde ocurrieron    las violaciones masivas del VaR al 99%. Crearemos una etiqueta binaria automática: 
Clase 0 (Régimen de Estabilidad/Calma) y Clase 1 (Régimen de Ruptura Estocástica / Pánico).

 El XGBoost será entrenado para predecir la probabilidad de que el Peso Mexicano salte a la Clase 1 en las próximas 24 horas. 
 
 La Ingeniería de Características Avanzada (Features / X): En lugar de meter variables crudas, alimentaremos al XGBoost con el ecosistema complejo que diagnosticamos en los cuadernos anteriores, uniendo las tres fuerzas macroeconómicas;
 - Features del Peso: El vector de volatilidad condicional diaria calculada por GARCH (\(\sigma _{t}\)) y los residuos puros del ARIMA.
 
 - Features Globales (La justificación de Granger): Dado que en el Notebook 1 se demostró mediante el Test de Granger que el pánico global (VIX) y las tasas americanas (Tasa_Fed_10Y) causan dinámicamente al Peso Mexicano con diferentes estructuras de tiempo, crearemos Vectores de Rezagos Temporales (Lag Features) del Lag 1 al Lag 5 de ambas variables.
 
 - Features No Lineales: Calcularemos las Covarianzas y Correlaciones Móviles de Spearman de 21 días para enseñarle al algoritmo cómo se está acoplando la energía del miedo global con el canal cambiario local antes de que estalle la crisis.
 
El Impacto Operativo de Negocio:

- Al finalizar este cuaderno, el corporativo no solo tendrá un escudo pasivo que le dice cuánto dinero puede perder (el VaR del Notebook 2), sino un Sistema de Alerta Temprana Algorítmico (Early Warning System).

Si el XGBoost detecta que la combinación de la volatilidad del GARCH, la latencia de 48 horas del VIX y la gravedad de las tasas elevan la probabilidad de pánico por encima de un umbral crítico, la empresa recibirá una señal automatizada para ejecutar coberturas cambiarias de forma proactiva, salvando el capital antes de que ocurra la perforación del mercado 

In [1]:
pip install xgboost scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 6.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import numpy as np

In [13]:
df_var = pd.read_csv('matriz_volatilidad_var_garch.csv', index_col='Date', parse_dates=True)
df_analisis = pd.read_csv('data_analisis_logs.csv', index_col='Date', parse_dates=True)
df_estacionario = pd.read_csv('data_estacionario.csv', index_col='Date', parse_dates=True)

In [14]:
df_estacionario.head()

,USD_MXN_log,Tasa_Fed_10Y_log,VIX_log
Date,,,
2021-01-05,0.004773,0.040604,3.232384
2021-01-06,-0.002020,0.087186,3.221672
2021-01-07,-0.012459,0.027451,3.107721
2021-01-08,0.017366,0.031253,3.070840
2021-01-11,0.002516,0.024141,3.181382


In [10]:
df_var.head()

,USD_MXN_real_log,volatilidad_garch,VaR_Sup_95,VaR_Inf_95,VaR_Sup_99,VaR_Inf_99
Date,,,,,,
2021-01-05,2.992851,0.008617,3.009597,2.976105,3.019936,2.965765
2021-01-06,2.990830,0.008012,3.006400,2.975260,3.016014,2.965647
2021-01-07,2.978371,0.007206,2.992376,2.964366,3.001023,2.955719
2021-01-08,2.995737,0.008297,3.011862,2.979612,3.021818,2.969656
2021-01-11,2.998254,0.010706,3.019059,2.977448,3.031905,2.964602


In [12]:
df_analisis.head()

,USD_MXN_log,Tasa_Fed_10Y_log,VIX_log
Date,,,
2021-01-04,2.988078,-0.086648,3.294725
2021-01-05,2.992851,-0.046044,3.232384
2021-01-06,2.990830,0.041142,3.221672
2021-01-07,2.978371,0.068593,3.107721
2021-01-08,2.995737,0.099845,3.070840


En Machine Learning de series de tiempo, el algoritmo no sabe qué día es hoy ni qué pasó ayer [finance]. Él solo ve renglones aislados [finance]. Por lo tanto, tu obligación como ingeniero es "abrirle los ojos" al pasado creando columnas nuevas que representen el tiempo [finance]. A esto le llamamos Lag Features (Vectores de Rezagos) [finance].

In [15]:
# 1. Inicializamos la tabla maestra usando el índice limpio de tu df_var (o df_estacionario, da igual porque coinciden)
X_master = pd.DataFrame(index=df_var.index)

# 2. Inyectamos la Volatilidad GARCH (Desde tu df_var)
X_master['volatilidad_garch'] = df_var['volatilidad_garch']

# 3. Inyectamos los Rendimientos Estacionarios del Peso (Desde tu df_estacionario)
X_master['rendimientos_usd'] = df_estacionario['USD_MXN_log']

# 4. INGENIERÍA DE REZAGOS (LAG FEATURES) DESDE EL DATA ESTACIONARIO PURO
# Creamos del lag 1 al lag 5 el pasado del VIX y las Tasas usando 'df_estacionario'
for lag in range(1, 6):
    X_master[f'vix_lag_{lag}'] = df_estacionario['VIX_log'].shift(lag)
    X_master[f'tasa_fed_lag_{lag}'] = df_estacionario['Tasa_Fed_10Y_log'].shift(lag)

# 5. INGENIERÍA DE CARACTERÍSTICAS NO LINEALES: CORRELACIONES MÓVILES DE 21 DÍAS
# Cruzamos las dos series de tu df_estacionario para evaluar el acoplamiento mensual
X_master['correlacion_vix_usd_21d'] = df_estacionario['VIX_log'].rolling(window=21).corr(df_estacionario['USD_MXN_log'])
X_master['correlacion_tasa_usd_21d'] = df_estacionario['Tasa_Fed_10Y_log'].rolling(window=21).corr(df_estacionario['USD_MXN_log'])

# 6. Purgamos los nulos (NaN) generados obligatoriamente por el rezago del rolling de 21 días
X_master = X_master.dropna()

print("============ MATRIZ DE CARACTERÍSTICAS COMPLETA Y AUDITADA ============")
print(f"Dimensiones de la matriz (Filas, Columnas): {X_master.shape}")
print("\nEcosistema de características listo para alimentar al XGBoost:")
print(X_master.columns.tolist())


============ MATRIZ DE CARACTERÍSTICAS COMPLETA Y AUDITADA ============
Dimensiones de la matriz (Filas, Columnas): (1368, 14)

Ecosistema de características listo para alimentar al XGBoost:
['volatilidad_garch', 'rendimientos_usd', 'vix_lag_1', 'tasa_fed_lag_1', 'vix_lag_2', 'tasa_fed_lag_2', 'vix_lag_3', 'tasa_fed_lag_3', 'vix_lag_4', 'tasa_fed_lag_4', 'vix_lag_5', 'tasa_fed_lag_5', 'correlacion_vix_usd_21d', 'correlacion_tasa_usd_21d']


2.0 Especificación de la Variable Objetivo (Target) y Desactivación del Desbalance Extremo"Habiendo consolidado la matriz de características predictoras (X_master) con 14 dimensiones estacionarias y de alta frecuencia [finance], el pipeline cuantitativo requiere la construcción de la Variable Objetivo o Vector Target (\(y\)) [finance]. Siguiendo los principios de la física de sistemas complejos, el algoritmo XGBoost no se entrenará para realizar predicciones lineales de precios [finance]. En su lugar, resolverá un problema de Clasificación Binaria de Alta Dimensión enfocado en anticipar transiciones de fase repentinas hacia regímenes de pánico macroeconómico global [finance].📐 El Dilema de la Modelación: ¿Por qué entrenar al 95% si el Escudo opera al 99%?Un error crítico de diseño en la Ciencia de Datos Financieros es extrapolar rígidamente los umbrales regulatorios al entrenamiento de algoritmos de Machine Learning [finance]. Si construyéramos la etiqueta target utilizando la frontera estricta del VaR al 99% validada en el Notebook 2, la base de datos exhibiría únicamente 10 violaciones reales a lo largo de 5 años (2021-2026) [finance].Estadísticamente, esto representaría un desbalance extremo de clases inferior al 0.7% [finance]. Ante una escasez de muestras de esta magnitud, un clasificador probabilístico basado en árboles de decisión (como XGBoost) sufre un colapso de sesgo destructivo [finance]: el optimizador aprende que la forma más eficiente de maximizar la función de pérdida y obtener un Accuracy artificial del 99.3% es volverse perezoso y predecir sistemáticamente la Clase 0 (Calma), volviéndose completamente ciego e inútil ante el peligro real [finance].Para otorgarle estabilidad numérica y densidad estadística al algoritmo, la variable objetivo se calibra utilizando la frontera superior de devaluación del VaR al 95% [finance]. Esta decisión estratégica se fundamenta en dos pilares de ingeniería de riesgo [finance]:Estabilidad de los Gradientes de Aprendizaje: Bajar el umbral de sensibilidad incrementa el número de violaciones observadas en la historia de la muestra, otorgándole al XGBoost el volumen de ejemplos necesarios para mapear con precisión matemática los patrones sutiles, las correlaciones móviles y las latencias de 48 horas del VIX que anuncian la tormenta antes de que ocurra [finance].Lógica Corporativa de Alerta Temprana (Early Warning System): Arquitectónicamente, el portafolio separa el rol de sus componentes [finance]. El VaR al 99% del Notebook 2 actúa como el escudo de defensa de capital final bajo normas internacionales de Basilea [2014, UCI, finance]. El XGBoost del Notebook 3 se convierte en el Termómetro de Pre-Alerta Preventiva [finance]. No tiene sentido financiero que la Inteligencia Artificial avise cuando el mercado ya perforó la barrera del 99% (momento en que la corporación ya incurrió en pérdidas críticas) [finance]. El XGBoost alertará cuando el sistema rompa la frontera del 95%, encendiendo una luz amarilla en la tesorería 24 horas antes para ejecutar coberturas de forma proactiva [finance].⛓️ Regla Matemáticas de Clasificación:Clase 0 (Régimen de Estabilidad / Calma): \(\text{USD\_MXN\_real\_log}_t \leq \text{VaR\_Sup\_95}_t\) ➔ El mercado opera dentro de las bandas normales [finance].Clase 1 (Régimen de Ruptura / Pre-Pánico): \(\text{USD\_MXN\_real\_log}_t > \text{VaR\_Sup\_95}_t\) ➔ Choque extremo de devaluación que activa el sistema de alerta

In [16]:
df_var.head()

,USD_MXN_real_log,volatilidad_garch,VaR_Sup_95,VaR_Inf_95,VaR_Sup_99,VaR_Inf_99
Date,,,,,,
2021-01-05,2.992851,0.008617,3.009597,2.976105,3.019936,2.965765
2021-01-06,2.990830,0.008012,3.006400,2.975260,3.016014,2.965647
2021-01-07,2.978371,0.007206,2.992376,2.964366,3.001023,2.955719
2021-01-08,2.995737,0.008297,3.011862,2.979612,3.021818,2.969656
2021-01-11,2.998254,0.010706,3.019059,2.977448,3.031905,2.964602


In [17]:
X_master.head()

,volatilidad_garch,rendimientos_usd,vix_lag_1,tasa_fed_lag_1,vix_lag_2,tasa_fed_lag_2,vix_lag_3,tasa_fed_lag_3,vix_lag_4,tasa_fed_lag_4,vix_lag_5,tasa_fed_lag_5,correlacion_vix_usd_21d,correlacion_tasa_usd_21d
Date,,,,,,,,,,,,,,
2021-02-03,0.009020,-0.011380,3.241029,0.025666,3.409166,-0.014747,3.499231,0.033492,3.408173,0.041532,3.616578,-0.025318,0.073843,0.073215
2021-02-04,0.009266,0.002945,3.131573,0.023257,3.241029,0.025666,3.409166,-0.014747,3.499231,0.033492,3.408173,0.041532,0.058383,0.051784
2021-02-05,0.008348,0.009369,3.080533,0.007049,3.131573,0.023257,3.241029,0.025666,3.409166,-0.014747,3.499231,0.033492,0.011984,0.142115
2021-02-08,0.008595,-0.013620,3.038313,0.026853,3.080533,0.007049,3.131573,0.023257,3.241029,0.025666,3.409166,-0.014747,0.041011,0.245347
2021-02-09,0.009502,-0.000414,3.055886,-0.008584,3.038313,0.026853,3.080533,0.007049,3.131573,0.023257,3.241029,0.025666,0.122952,0.169018


In [ ]:
# 1. Definimos la condición de ruptura: Días donde el precio real superó el techo del VaR al 95%
# (Usamos el VaR al 95% para tener una muestra de entrenamiento con suficientes ejemplos de la Clase 1)
condicion_panico = df_var['USD_MXN_real_log'] > df_var['VaR_Sup_95']

# 2. Fabricamos el vector target 'y' inicializándolo en cero (Clase 0: Calma)
y_full = pd.Series(0, index=df_var.index)

# 3. Asignamos un 1 (Clase 1: Pánico) a los días donde se cumplió la perforación estricta
y_full[condicion_panico] = 1

# 4. ALINEACIÓN CRUCIAL: Filtramos el vector target para que tenga exactamente el mismo calendario
# y tamaño que tu matriz 'X_master' (eliminando los primeros 20 días de NaNs del rolling)
y = y_full.loc[X_master.index]

print("============ AUDITORÍA DEL VECTOR TARGET (Y) ============")
print(f"Dimensiones finales de la variable objetivo y: {y.shape}")
print("\nDistribución real de Regímenes en tu base de datos (2021-2026):")
print(y.value_counts())

# Calculamos la proporción del desbalance de clases
proporcion_panico = (y.sum() / len(y)) * 100
print(f"\nPorcentaje de días en Régimen de Pánico (Clase 1): {proporcion_panico:.2f}%")
